In [1]:
import json 
import spacy
import scispacy
from scispacy.linking import EntityLinker
import pubmed_parser as pp
from tqdm import tqdm

nlp = spacy.load("en_core_sci_lg")
nlp.add_pipe("scispacy_linker", config={"resolve_abbreviations": True, "linker_name": "umls"})

nlp2 = spacy.load('en_ner_bc5cdr_md')
nlp2.add_pipe("scispacy_linker", config={"resolve_abbreviations": True, "linker_name": "umls"})

dicts_out = pp.parse_medline_xml('../data/pubmed_data/processed_data/processed_full_data.xml',
                                 year_info_only=False,
                                 nlm_category=False,
                                 author_list=False,
                                 reference_list=False)

/home/rgoli/.conda/envs/pytorch/lib/python3.8/site-packages/spacy/util.py:833: UserWarning: [W095] Model 'en_core_sci_lg' (0.4.0) was trained with spaCy v3.0 and may not be 100% compatible with the current version (3.2.4). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)
/home/rgoli/.local/lib/python3.8/site-packages/sklearn/base.py:324: UserWarning: Trying to unpickle estimator TfidfTransformer from version 0.20.3 when using version 1.0.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/modules/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/rgoli/.local/lib/python3.8/site-packages/sklearn/base.py:324: UserWarning: Trying to unpickle estimator TfidfVectorizer from version 0.20.3 w

In [2]:
## HoldOut 125 GS + 111 No/Little Abstract

pmids_HO = set()
with open('../data/HoldOut_Total.txt','r') as f:
    for line in f.readlines():
        pmids_HO.add(line.strip())
print("Hold Out Data: ",len(pmids_HO))

Hold Out Data:  236


In [3]:
N=len(dicts_out)
i=0
j=0
while i<N:
    if dicts_out[i]['pmid'] not in pmids_HO:
        dicts_out[j]=dicts_out[i]
        j+=1
    i+=1
dicts_out[j:]=[]

print("Full Data for further processing: ",len(dicts_out))

Full Data for further processing:  3100


In [4]:
comb_arr=[]
pmid_arr =[]
for paper in dicts_out:
    comb_arr.append(paper['title']+' '+paper['abstract'])
    pmid_arr.append(paper['pmid'])
    
idx=0
processed_texts = {}
scispacy_kw = {}
no_kws_trimmed =[]
no_kws=[]
post_no_kws =[]
outlier_kws=[
    'http','www','university','department','antibiotic','antimicrobial','institute','ministry', 'pubmed',
    '.gov','.org','.com','.edu','.net',
    'city','disease','injury','trauma','syndrome','country','national','regimen','swelling',
    'surgery', 'medication','infection','stroke','diabetes','bleeding','comorbid','java','python',
    "united states",'united kingdom','india','china','germany','france','ghana','australia','italy','england','japan',
    'english','spanish','french','british','spain'
    ]

remove_sent_no_outliers = []
with tqdm(nlp.pipe(comb_arr),total=len(comb_arr)) as pbar:
    for doc in pbar:
        sents = [[token.orth_ for token in sent if not token.is_punct | token.is_space] for sent in doc.sents]
        if len(sents)<4:
            remove_sent_no_outliers.append(idx)
            idx+=1
            continue
        text = [token.orth_ for token in doc if not token.is_punct | token.is_space] 
        rule_out_ents = [str(_) for _ in nlp2(comb_arr[idx]).ents]
        rule_out_ents = [_ for _ in rule_out_ents if not _.isupper()]
        kws = [str(_) for _ in doc.ents]
        # print('-'*50)
        # print(len(kws))
        temp_kws_len = len(kws)
        # print(kws)
        if rule_out_ents==[]:
            kws = [_ for _ in kws]
        else:
            kws = [_ for _ in kws if _ not in rule_out_ents]
        # print(kws)
        kws = [_ for _ in kws if all([True if n_ not in _.lower() else False for n_ in outlier_kws])]
        no_kws_trimmed.append(temp_kws_len-len(kws))
        no_kws.append(temp_kws_len)
        post_no_kws.append(len(kws))
        # print(len(kws))
        # print(rule_out_ents)
        # print(kws)
        # print('-'*50)
        kws.sort(reverse=True, key=len)
        
        dicts_out[idx]['fullText'] = text
        dicts_out[idx]['sentences'] = sents
        dicts_out[idx]["keywords"]=kws=kws
        
        idx+=1
        pbar.set_description("The %s document is processed" % (idx + 1))

## Filter Docs with less than minimum senetences
i=0
j=0
print("Total Articles removed: ",len(remove_sent_no_outliers))
print([pmid_arr[x] for x in remove_sent_no_outliers])
while i<idx:
    if i not in remove_sent_no_outliers:
        dicts_out[j]=dicts_out[i]
        j+=1
    i+=1
dicts_out[j:]=[]

dataset_length = len(dicts_out)

print("Total Dataset Length: ",dataset_length)
print("Train Dataset Length: ",dataset_length//3)
print("Test  Dataset Length: ",dataset_length-dataset_length//3)

with open("GNA_Files/train.json", "w") as outfile:
    json.dump(dicts_out[:dataset_length//3], outfile)

with open("GNA_Files/test.json", "w") as outfile:
    json.dump(dicts_out[dataset_length//3:], outfile)

print("Total KWS generated:", sum(no_kws))
print("Avg. KWS generated/abstract:", sum(no_kws)/len(no_kws))

print("Total KWS trimmed: ",sum(no_kws_trimmed))
print("Avg. KWS trimmed:", sum(no_kws_trimmed)/len(no_kws_trimmed))

print("Total KWS - post processing: ",sum(post_no_kws))
print("Avg. KWS - post processing:", sum(post_no_kws)/len(post_no_kws))

  0%|          | 0/3100 [00:00<?, ?it/s]/home/rgoli/.conda/envs/pytorch/lib/python3.8/site-packages/scispacy/candidate_generation.py:284: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  extended_neighbors[empty_vectors_boolean_flags] = numpy.array(neighbors)[:-1]
/home/rgoli/.conda/envs/pytorch/lib/python3.8/site-packages/scispacy/candidate_generation.py:285: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  extended_distances[empty_vectors_boolean_flags] = numpy.array(distances)[:-1]
The 3101 document is processed: 100%|██████████| 3100/3100 [04:42<00:00,

Total Articles removed:  48
['34042851', '31349252', '31225983', '30399617', '28269917', '28086985', '27762202', '27235214', '26262257', '26262255', '23920886', '23920721', '23861123', '23851288', '22929351', '22219234', '22060071', '21295751', '21095686', '20592007', '20501532', '20407168', '18999221', '18999012', '18998900', '18998875', '18694272', '18694237', '18580428', '18487843', '17945725', '17911954', '17238710', '17238621', '17238566', '17238503', '16779188', '15846970', '15770911', '15455842', '14728543', '12365297', '11813521', '10587916', '10338700', '9700425', '10177918', '10177693']
Total Dataset Length:  3052
Train Dataset Length:  1017
Test  Dataset Length:  2035
Total KWS generated: 224833
Avg. KWS generated/abstract: 73.66743119266054
Total KWS trimmed:  12175
Avg. KWS trimmed: 3.989187418086501
Total KWS - post processing:  212658
Avg. KWS - post processing: 69.67824377457406


In [28]:
train_data = dicts_out[:dataset_length//3]
test_data = dicts_out[dataset_length//3:]

In [29]:
train_corpus_sentences = []
for x in train_data:
    train_corpus_sentences.extend(x['sentences'])
print("Total Sentences in Train Corpus: ", len(train_corpus_sentences))

test_corpus_sentences = []
for x in test_data:
    test_corpus_sentences.extend(x['sentences'])
print("Total Sentences in Test Corpus: ", len(test_corpus_sentences))

Total Sentences in Train Corpus:  12023
Total Sentences in Test Corpus:  20555


In [30]:
print('Sample: ', train_corpus_sentences[0])

Sample:  ['Continuous', 'cardiorespiratory', 'monitoring', 'is', 'a', 'dominant', 'source', 'of', 'predictive', 'signal', 'in', 'machine', 'learning', 'for', 'risk', 'stratification', 'and', 'clinical', 'decision', 'support']


In [31]:
import gensim

model = gensim.models.Word2Vec(sentences=train_corpus_sentences,size=300,window=5,min_count=0,workers=10,sg=1,seed=42,compute_loss=True)

In [33]:
model.save('GNA_Files/word2vec.embedding')

In [41]:
model = gensim.models.Word2Vec.load('GNA_Files/word2vec.embedding')
model.wv.most_similar(positive=['clinical','decision'], topn = 10)

[('computerized', 0.9211411476135254),
 ('computerised', 0.9013410806655884),
 ('computer', 0.9006768465042114),
 ('CPOE', 0.8991823792457581),
 ('decision-support', 0.8988824486732483),
 ('point-of-care', 0.8986619710922241),
 ('guideline-based', 0.8960883021354675),
 ('phone-based', 0.8882958889007568),
 ('real-time', 0.8872377276420593),
 ('automated', 0.8859662413597107)]

In [43]:
model.wv.most_similar(positive=['cardiorespiratory'], topn = 10)

[('cross-referenced', 0.9977792501449585),
 ('trachomatis', 0.9977713823318481),
 ('permissioned', 0.997490644454956),
 ('Grades', 0.997468113899231),
 ('pillar', 0.9974541664123535),
 ('changepoint', 0.9974361062049866),
 ('Utah', 0.9974315762519836),
 ('dose/drug', 0.9974045157432556),
 ('Co-management', 0.9973499774932861),
 ('metric-related', 0.9972480535507202)]

In [44]:
model.wv.most_similar(positive=['machine','learning'], topn = 10)

[('fuzzy', 0.9588775634765625),
 ('network', 0.9555433988571167),
 ('techniques', 0.9498676061630249),
 ('artificial', 0.9463967680931091),
 ('logic', 0.9349775910377502),
 ('algorithm', 0.9342019557952881),
 ('vector', 0.9331274032592773),
 ('modeling', 0.931676983833313),
 ('ML', 0.9294275641441345),
 ('neural', 0.928849995136261)]